# 02. NLP Preparation for Recruitment Text

This notebook demonstrates basic NLP operations on a synthetic resume and job description.

## 1. Introduction to NLP

Natural Language Processing converts unstructured text into representations that software can inspect.

## 2. Why NLP is useful for recruitment

Resumes and job descriptions contain skills, roles, education, and experience in varied wording. Cleaning and tokenization make simple comparisons possible.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import re
from collections import Counter
import pandas as pd
import nltk

resume_path = ROOT / "data" / "sample_data" / "resumes.csv"
jd_path = ROOT / "data" / "sample_data" / "job_descriptions.csv"
resumes = pd.read_csv(resume_path)
jds = pd.read_csv(jd_path)
print("NLTK version:", nltk.__version__)

NLTK version: 3.10.3


## 3. Load a sample resume and JD

In [2]:
sample_resume = resumes.loc[0, "resume_text"]
sample_jd = jds.loc[0, "job_description"]
print("Resume:", sample_resume)
print("JD:", sample_jd)

Resume: Synthetic Data Analyst candidate with skills Python;SQL and 2 years of experience.
JD: Synthetic Data Analyst position requiring Python;SQL.


## 4. Raw text inspection

In [3]:
print("Resume characters:", len(sample_resume))
print("Resume words before cleaning:", len(sample_resume.split()))
print("JD characters:", len(sample_jd))

Resume characters: 82
Resume words before cleaning: 12
JD characters: 53


## 5. Text normalization

In [4]:
normalized_resume = " ".join(sample_resume.split())
normalized_jd = " ".join(sample_jd.split())
print(normalized_resume)

Synthetic Data Analyst candidate with skills Python;SQL and 2 years of experience.


## 6. Lowercase conversion

In [5]:
lower_resume = normalized_resume.lower()
lower_jd = normalized_jd.lower()
print(lower_resume)

synthetic data analyst candidate with skills python;sql and 2 years of experience.


## 7. Punctuation and special-character removal

In [6]:
punctuation_free_resume = re.sub(r"[^a-z0-9+#.\- ]", " ", lower_resume)
punctuation_free_resume = " ".join(punctuation_free_resume.split())
punctuation_free_jd = " ".join(re.sub(r"[^a-z0-9+#.\- ]", " ", lower_jd).split())
print(punctuation_free_resume)

synthetic data analyst candidate with skills python sql and 2 years of experience.


## 8. Tokenization

In [7]:
tokens = re.findall(r"[a-z0-9+#.\-]+", punctuation_free_resume)
jd_tokens = re.findall(r"[a-z0-9+#.\-]+", punctuation_free_jd)
print("Resume tokens:", tokens)
print("Token count:", len(tokens))

Resume tokens: ['synthetic', 'data', 'analyst', 'candidate', 'with', 'skills', 'python', 'sql', 'and', '2', 'years', 'of', 'experience.']
Token count: 13


## 9. Stopword removal

In [8]:
try:
    from nltk.corpus import stopwords
    stop_words = set(stopwords.words("english"))
    print("Using NLTK English stopwords:", len(stop_words))
except LookupError:
    stop_words = {"a", "an", "and", "are", "for", "in", "of", "the", "to", "with"}
    print("NLTK stopwords resource unavailable; using a small local fallback.")

content_tokens = [token for token in tokens if token not in stop_words]
print("After stopword removal:", content_tokens)

NLTK stopwords resource unavailable; using a small local fallback.
After stopword removal: ['synthetic', 'data', 'analyst', 'candidate', 'skills', 'python', 'sql', '2', 'years', 'experience.']


## 10. Optional stemming or lemmatization

In [9]:
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()
stems = [stemmer.stem(token) for token in content_tokens]
print(list(zip(content_tokens, stems)))

[('synthetic', 'synthet'), ('data', 'data'), ('analyst', 'analyst'), ('candidate', 'candid'), ('skills', 'skill'), ('python', 'python'), ('sql', 'sql'), ('2', '2'), ('years', 'year'), ('experience.', 'experience.')]


## 11. Word-frequency analysis

In [10]:
frequencies = Counter(content_tokens)
print("Most common resume terms:", frequencies.most_common(10))

Most common resume terms: [('synthetic', 1), ('data', 1), ('analyst', 1), ('candidate', 1), ('skills', 1), ('python', 1), ('sql', 1), ('2', 1), ('years', 1), ('experience.', 1)]


## 12. Before vs after preprocessing

In [11]:
print("Before:", sample_resume)
print("After:", " ".join(content_tokens))
print("Resume token reduction:", len(tokens), "to", len(content_tokens))

Before: Synthetic Data Analyst candidate with skills Python;SQL and 2 years of experience.
After: synthetic data analyst candidate skills python sql 2 years experience.
Resume token reduction: 13 to 10


## 13. Apply preprocessing to sample resume/JD data

In [12]:
def preprocess_text(text):
    lowered = str(text).lower()
    cleaned = " ".join(re.sub(r"[^a-z0-9+#.\- ]", " ", lowered).split())
    words = re.findall(r"[a-z0-9+#.\-]+", cleaned)
    return " ".join(word for word in words if word not in stop_words)

resumes["notebook_cleaned_text"] = resumes["resume_text"].map(preprocess_text)
jds["notebook_cleaned_text"] = jds["job_description"].map(preprocess_text)
print(resumes[["candidate_label", "notebook_cleaned_text"]].head(3))
print(jds[["jd_id", "notebook_cleaned_text"]].head(3))

  candidate_label                              notebook_cleaned_text
0   Candidate_001  synthetic data analyst candidate skills python...
1   Candidate_002  synthetic ml engineer candidate skills python ...
2   Candidate_003  synthetic nlp engineer candidate skills sql st...
    jd_id                              notebook_cleaned_text
0  JD_001  synthetic data analyst position requiring pyth...
1  JD_002  synthetic ml engineer position requiring pytho...
2  JD_003  synthetic nlp engineer position requiring sql ...


## 14. Store cleaned text

In [13]:
processed_resume_path = ROOT / "data" / "processed" / "resumes_nlp_notebook.csv"
processed_jd_path = ROOT / "data" / "processed" / "job_descriptions_nlp_notebook.csv"
resumes.to_csv(processed_resume_path, index=False)
jds.to_csv(processed_jd_path, index=False)
print("Saved:", processed_resume_path.name, "and", processed_jd_path.name)

Saved: resumes_nlp_notebook.csv and job_descriptions_nlp_notebook.csv


## 15. Small NLP practice exercise

In [14]:
practice = "Python Python SQL analytics"
practice_tokens = re.findall(r"[a-z]+", practice.lower())
print(Counter(practice_tokens).most_common())

[('python', 2), ('sql', 1), ('analytics', 1)]


## 16. Key Learning Outcomes

- Normalize, lowercase, and clean recruitment text.
- Tokenize text and remove stopwords with an NLTK resource or documented fallback.
- Apply stemming and word-frequency analysis.
- Store cleaned resume and JD text for later vectorization.